In [1]:
import aiida
from aiida import orm, load_profile
from aiida.storage.sqlite_zip.backend import SqliteZipBackend
from aiida_wannier90_workflows.workflows.optimize import Wannier90OptimizeWorkChain
from tabulate import tabulate

In [2]:
! verdi profile setup core.sqlite_zip --filepath ../mc3d_wannier_3materials.aiida -p mc3d-pbesol-v1-wannier -n

Usage: verdi profile setup core.sqlite_zip [OPTIONS]
Try 'verdi profile setup core.sqlite_zip --help' for help.

Error: Invalid value for '-p' / '--profile-name': the profile `mc3d-pbesol-v1-wannier` already exists


In [3]:
AIIDA_PROFILE_NAME = "mc3d-pbesol-v1-wannier"
aiida.load_profile(AIIDA_PROFILE_NAME)

Profile<uuid='d128bf5f00ad4922abd04e2677fc326d' name='mc3d-pbesol-v1-wannier'>

In [4]:
# work chains are here.
w90_workchains = orm.QueryBuilder().append(Wannier90OptimizeWorkChain).all(flat=True)
w90_workchains

[<WorkChainNode: uuid: 94422a15-d26d-4d95-a0c8-9b4fb7f07e91 (pk: 6607169) (aiida.workflows:wannier90_workflows.optimize)>,
 <WorkChainNode: uuid: d45b0108-da1f-40d6-98ad-0f9d67186e43 (pk: 9892286) (aiida.workflows:wannier90_workflows.optimize)>,
 <WorkChainNode: uuid: 3ee04a62-2d61-4f0e-8cee-cb77967ef330 (pk: 13171598) (aiida.workflows:wannier90_workflows.optimize)>]

In [5]:
# load the pbesol-v1 base json to map structure UUIDs to this uuid.
import json

with open("../../core/base.json", "r") as f:
    mc3d_data = json.load(f)

uuid_to_mc3d = {}

for mc3d_id, entry in mc3d_data.items():
    structure_uuid = entry["general"]["structure_uuid"]
    uuid_to_mc3d[structure_uuid] = mc3d_id

print(uuid_to_mc3d)

{'85cc46fe-ea70-45a0-8c60-36f5e6407cfa': 'mc3d-10', '24116963-2e71-4e96-837c-8a4edcd06e49': 'mc3d-10000', '5e70c3fc-9486-4fac-bcdc-1736ce9382cf': 'mc3d-10004', 'd14a7a03-8b5f-4a00-a579-2794f905f5ee': 'mc3d-10006', '8ca4af98-edf2-4eb5-94a1-77e418a0dde4': 'mc3d-10007', '82280140-4e85-42ad-848d-ccbb73559e58': 'mc3d-10008', 'c48bed29-7b32-46dc-9a26-f0c3d2ecaba4': 'mc3d-1001', 'd6f34558-eb00-4a30-a1a4-47949029ee8b': 'mc3d-10011', '9eba8266-92f7-4aca-b8d0-588dcd546c2d': 'mc3d-10015', '72eb0a33-8227-4a30-910e-5109bef211a6': 'mc3d-10016', 'f64c3bb9-36cd-475a-a512-e8eee8dd6832': 'mc3d-10017', '1a34e1a1-ae48-4cd1-9b04-2cda4996ed5b': 'mc3d-10019', '0cd0a9a2-2a9a-4cfc-8579-f791d9347b2f': 'mc3d-10021', '9360860e-23c4-4e4d-acbf-2d73814660bc': 'mc3d-10023', 'a2e236d9-093b-4e10-8efc-853398800abc': 'mc3d-10030', 'cb123212-10b8-4e6a-9f8e-1a92ab711f28': 'mc3d-10031', '3979a4be-9178-44b4-ab21-142ab4bd3ada': 'mc3d-10044', 'b2c57d81-05bf-437c-bd45-8f0694a2104f': 'mc3d-10048', '5bdac0ef-26cf-4b4f-b7e9-c586b9

In [9]:
# Step 2: Build simplified mapping
final_mapping = {}

for w90 in w90_workchains:
    # Get the structure UUID
    structure_uuid = w90.inputs.nscf.pw.parent_folder.creator.inputs.stash_data.creator.caller.caller.inputs.structure.uuid
    
    mc3d_id = uuid_to_mc3d.get(structure_uuid)
    if mc3d_id is None:
        # skip or log missing matches
        continue

    # Get the relevant UUIDs
    pwband = w90.inputs.nscf.pw.parent_folder.creator.inputs.stash_data.creator.caller.caller
    # print(w90.inputs.nscf.pw.parent_folder.creator.inputs.stash_data.creator.uuid)
    # print(w90.inputs.nscf.pw.parent_folder.creator.inputs.stash_data.creator.caller.uuid)
    band_pw = pwband.outputs.band_structure
    band_w90 = w90.outputs.band_structure

    fermi = pwband.outputs.scf_parameters.get('fermi_energy')

    cj_stash = "".join(w90.outputs.wannier90_plot__remote_stash.target_basepath.split("/")[-3:])

    # w90 information
    out_dict = w90.outputs.wannier90_optimal.output_parameters.get_dict()

    wf_data = [] # array 
    for wf in out_dict.get("wannier_functions_output", []):
        wf_data.append({
            "index": wf["wf_ids"],
            "center": wf["wf_centres"],
            "spread": wf["wf_spreads"],
        })

    # Wannier90 info
    number_wfs = out_dict.get('number_wfs')
    final_mapping[mc3d_id] = {
        "w90_wc_uuid": w90.uuid,
        "pw_band_uuid": band_pw.uuid,
        "w90_band_uuid": band_w90.uuid,
        "cj_stash_uuid": cj_stash,
        "fermi_energy": fermi,
        "wf_info": {
            "num_wf": out_dict.get('number_wfs'),
            "omega_i": out_dict.get('Omega_I'),
            "omega_d": out_dict.get('Omega_D'),
            "omega_od": out_dict.get('Omega_OD'),
            "omega_sum": out_dict.get('Omega_I') + out_dict.get('Omega_D') + out_dict.get('Omega_OD'),
            "wf_array": wf_data
        }
    }

# Step 3: Write to JSON file
with open("../base.json", "w") as f:
    json.dump(final_mapping, f, indent=2)

print("Final mapping written to mc3d_to_uuids.json")

/tmp/ipykernel_3783951/1741404338.py:22: AiidaDeprecationWarning: dereferencing nodes with links containing double underscores is deprecated, simply replace the double underscores with a single dot instead. For example: 
`node.inputs.some__label` can be written as `node.inputs.some.label` instead.
 (this will be removed in v3)
  cj_stash = "".join(w90.outputs.wannier90_plot__remote_stash.target_basepath.split("/")[-3:])
/tmp/ipykernel_3783951/1741404338.py:22: AiidaDeprecationWarning: dereferencing nodes with links containing double underscores is deprecated, simply replace the double underscores with a single dot instead. For example: 
`node.inputs.some__label` can be written as `node.inputs.some.label` instead.
 (this will be removed in v3)
  cj_stash = "".join(w90.outputs.wannier90_plot__remote_stash.target_basepath.split("/")[-3:])


Final mapping written to mc3d_to_uuids.json


/tmp/ipykernel_3783951/1741404338.py:22: AiidaDeprecationWarning: dereferencing nodes with links containing double underscores is deprecated, simply replace the double underscores with a single dot instead. For example: 
`node.inputs.some__label` can be written as `node.inputs.some.label` instead.
 (this will be removed in v3)
  cj_stash = "".join(w90.outputs.wannier90_plot__remote_stash.target_basepath.split("/")[-3:])
